In [ ]:
import pandas as pd
import numpy as np

# ---------------------------
# 1. Load dataset
# ---------------------------
df = pd.read_csv("/content/Merged_Disasters_Cleaned_Dataset.csv")


# =====================================================
#             A. PREPARATORY CALCULATIONS
# =====================================================

# -------- GDP per capita --------
# Your dataset already contains:
# "GDP per capita, PPP (constant 2021 international $)"
df["GDP_Per_Capita"] = df["GDP per capita, PPP (constant 2021 international $)"]

# -------- Fatalities per million --------
# Death_Rate appears to be deaths per 100k.
df["Fatalities_Per_Million"] = df["Death_Rate"] * 10

# -------- Affected population % --------
# Affected_Rate is also per 100k.
df["Affected_Percent"] = (df["Affected_Rate"] / 100_000) * 100

# -------- Severity Weight --------
# You already have a final calculated severity:
df["Severity_Weight_Final"] = df["Severity_Weight"]


# =====================================================
#             B. DISASTER IMPACT INDEX (DII)
# =====================================================

df["DII"] = (
    (df["Fatalities_Per_Million"] + df["Affected_Percent"])
    / df["GDP_Per_Capita"]
) * df["Severity_Weight_Final"]


# =====================================================
#            C. RESILIENCE RECOVERY SCORE (RRS)
# =====================================================

# Identify governance index column if present
gov_columns = [c for c in df.columns if "Gov" in c]
if len(gov_columns) > 0:
    gov_col = gov_columns[0]
else:
    # If governance index missing → assign neutral value for formula
    df["GovIndex"] = 0.5
    gov_col = "GovIndex"

# Avoid division by zero in recovery years:
df["T_Recovery_Years_Adj"] = df["T_Recovery_Years"].replace(0, np.nan)

df["RRS"] = (
    (df["GDP_Growth_Rate"] - df["Avg_Growth_Rate"]) / df["T_Recovery_Years_Adj"]
) + (df["Human Development Index"] + df[gov_col]) / 2


# =====================================================
#          D. COMPOSITE RESILIENCE INDEX (CRI)
# =====================================================

# ---- Adaptive Capacity (A) ----
mapping = {"Low": 1, "Medium": 2, "High": 3}
df["Adaptive_Capacity_Score"] = df["Adaptive_Capacity_Level"].map(mapping)

# We could optionally enhance A using infrastructure components:
df["Adaptive_Capacity_Final"] = (
    df["Adaptive_Capacity_Score"]
    + df["Urbanization_Capacity"]
    + df["Water_Capacity"]
    + df["HDI_Capacity"]
) / 4   # Normalize to scale

# ---- Exposure (E) ----
df["Exposure"] = df["Disaster_Intensity"]

# ---- Vulnerability (V) ----
df["Vulnerability"] = df["Death_Rate"] + df["GDP_Loss_Percent"]

# ---- Final CRI ----
df["CRI"] = df["Adaptive_Capacity_Final"] / (
    df["Exposure"] * df["Vulnerability"].replace(0, np.nan)
)


# =====================================================
#         E. Export updated dataset with indices
# =====================================================

df.to_csv("Disaster_Resilience_Indices_Output.csv", index=False)

print("All indices computed successfully!")
df[["Entity", "Year", "DII", "RRS", "CRI"]].head()


/tmp/ipython-input-2039374526.py:7: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/Merged_Disasters_Cleaned_Dataset.csv")


All indices computed successfully!


,Entity,Year,DII,RRS,CRI
0,Afghanistan,2000,0.011676,NaN,0.009817
1,Afghanistan,2000,0.000368,NaN,0.315097
2,Afghanistan,2000,0.000441,NaN,0.264209
3,Afghanistan,2000,0.000347,NaN,0.335577
4,Afghanistan,2000,0.005888,-2.748761,0.061884


In [ ]:
df[df["RRS"].isna()][["GDP_Growth_Rate", "Avg_Growth_Rate", "T_Recovery_Years", "Human Development Index", "GovIndex"]].head(20)


,GDP_Growth_Rate,Avg_Growth_Rate,T_Recovery_Years,Human Development Index,GovIndex
0,0.0,0.04951,0.0,0.351,0.5
1,0.0,0.04951,0.0,0.351,0.5
2,0.0,0.04951,0.0,0.351,0.5
3,0.0,0.04951,0.0,0.351,0.5
5,0.0,0.04951,0.0,0.351,0.5
6,0.0,0.04951,0.0,0.351,0.5
7,0.0,0.04951,0.0,0.351,0.5
8,0.0,0.04951,0.0,0.351,0.5
10,0.0,0.04951,0.0,0.351,0.5
11,0.0,0.04951,0.0,0.351,0.5


In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/Disaster_Resilience_Indices_Output.csv")

# -----------------------------------------
# 1. Multi-Hazard Exposure Score
# -----------------------------------------
df["MultiHazard_Exposure"] = (
    df["Disaster_Intensity"] * df.get("Event_Count", 1)
)



# -----------------------------------------
# 3. Human Severity Score (HSS)
# -----------------------------------------
df["Human_Severity_Score"] = (
    0.7 * df["Death_Rate"] +
    0.3 * df["Affected_Rate"]
)

# -----------------------------------------
# 4. Urban Impact Ratio (UIR)
# -----------------------------------------
df["Urban_Impact_Ratio"] = (
    df["Total Affected"] / df["Urban population"].replace(0, np.nan)
)

# handling missing values
df["T_Recovery_Years_Adj"] = df["T_Recovery_Years_Adj"].fillna(1)

df["RRS"] = df.groupby(["Region", "Disaster Type"])["RRS"].transform(
    lambda x: x.fillna(x.mean())
)

# Step 2
df["RRS"] = df.groupby("Region")["RRS"].transform(
    lambda x: x.fillna(x.mean())
)

# Step 3
df["RRS"] = df["RRS"].fillna(df["RRS"].mean())

print("Top 5 engineered features added successfully!")
df.to_csv("DAV_dataset.csv", index=False)
print("Saved as DAV_dataset.csv")


/tmp/ipython-input-592638563.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/Disaster_Resilience_Indices_Output.csv")


Top 5 engineered features added successfully!
Saved as DAV_dataset.csv
